In [ ]:
# Assessing geospatial distribution of population statistics.

Assessing Geospatial Distribution of Population Statistics: Population Density
Script includes :
 - modules imported at the beginning of the script
 - joining of census data to geodatabase data zones
 - statistics
 - mapping outputs

External module import:

In [ ]:
%matplotlib widget
import os 
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from cartopy.feature import ShapelyFeature
import cartopy.crs as ccrs
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

plt.ion() # make the plotting interactive

Import Data
- Census data csv file
- Data zone shapefile

In [ ]:
zones = gpd.read_file('data_files/DZ2021.shp') # import data zones shapefile as zones
census = pd.read_csv('data_files/Census.csv') # import census data csv file as census


Join Census Data to Data Zones
DZ_2021_Code from census will join with DZ2021_cd from zones

In [ ]:
zones.head() # identify header that will join with census data

In [ ]:
census.head() # identify header that will join with census data

In [ ]:
census = census.rename({'DZ_2021_Code' : 'DZ2021_cd'}, axis=1) #rename census.DZ_2021_Code to DZ2021_cd to match zones
census.head() # error check

In [ ]:
census_zones = gpd.GeoDataFrame(pd.merge(zones, census, on='DZ2021_cd', how='outer').fillna(0)) # join census and zones based on common column DZ2021_cd using an outer join and filling NA values.

census_zones # check output 

Calculate Population Density

In [ ]:
census_zones.to_file('census_zones.shp') # export to shapefile

In [ ]:
if census_zones.crs is None: # ensure census_zones has projected CRS for area calculation 
    raise ValueError ("No CRS found. Please set CRS before calculating area.")
    
if census_zones.crs.is_geographic: #reproject to CRS in meters (e.g. mercator) - Northern Ireland EPSG:29902
    census_zones=census_zones.to_crs(epsg=29902)

In [ ]:
for ind, row in census_zones.iterrows():
    census_zones.loc[ind,'Area_KM2']=row['geometry'].area/1000000,
    # creates new column 'Area_KM2' by taking geometry m and converting it to area in km2)

In [ ]:
def add_area_km2(gdf):
    """


    """
    if 'geometry' not in gdf.columns:
        raise ValueError("GeoDataFrame must have a 'geometry' column.")
        gdf['Area_KM2']=gdf.geometry.area/1000000
        return gdf

census_zones = add_area_km2(census_zones)

In [ ]:
census_zones.columns # view column names 

In [ ]:
if "popn" not in census_zones.columns: # validate popn exists
    raise KeyError ("'popn' column not found in dataset")

census_zones['PopnDens']=census_zones['popn']/census_zones['Area_KM2'] # create population density column from  popn / area_km2

print(census_zones.head())

Statistics 

The following section of code identifies various statistics of the data
- mean
- median
- minimum values
- maximum values
- range

In addition, code identifies population density broken down into local government districts (LGD2014_nm).

In [ ]:
census_zones['PopnDens'].mean() # find mean of population density in Northern Ireland

In [ ]:
census_zones['PopnDens'].median() # find median of population density in Northern Ireland

In [ ]:
#find range of population density
popdens_data = census_zones['PopnDens'].dropna() # remove n/a values
min_val = popdens_data.min() # find minimum value in popndens column
max_val = popdens_data.max() # find maximum value in popndens column

print(f"range of {'PopnDens'}: min = {min_val}, max = {max_val}")

Mapping

The following section maps the population density of 

In [ ]:
# categorise population density into low, medium high where low is below 300 popn/km2, medium is 300-10000 popn/km2 and high is 10000+ popn/km2
def popdens_category(value):
       """
    Assign categories to population density values.

    Parameters
    ----------

    values : value
    values : value
        values assigned to categories
    Returns
    -------
    High : where value is greater than or equal to 10000
    Medium : where value is equal to or greater than 300 and less thaan 10000
    Low : where value is less than 300

        """
    if value < 300:
        return "Low"
    elif 300 <= value < 10000:
        return "Medium"
    else:
        return "High"

# Create a new column using map()
census_zones['PopDens_Category'] = census_zones['PopnDens'].map(popdens_category) #selects PopDens column and applies popdens function to each value, stored in new column PopDens_Category

In [ ]:
def generate_handles(labels, colors, edge='k', alpha=1):
    """
    Generate matplotlib patch handles to create a legend of each of the features in the map.

    Parameters
    ----------

    labels : list(str)
        the text labels of the features to add to the legend

    colors : list(matplotlib color)
        the colors used for each of the features included in the map.

    edge : matplotlib color (default: 'k')
        the color to use for the edge of the legend patches.

    alpha : float (default: 1.0)
        the alpha value to use for the legend patches.

    Returns
    -------

    handles : list(matplotlib.patches.Rectangle)
        the list of legend patches to pass to ax.legend()
    """
    lc = len(colors)  # get the length of the color list
    handles = [] # create an empty list
    for ii in range(len(labels)): # for each label and color pair that we're given, make an empty box to pass to our legend
        handles.append(mpatches.Rectangle((0, 0), 1, 1, facecolor=colors[ii % lc], edgecolor=edge, alpha=alpha))
    return handles

# adapted this question: https://stackoverflow.com/q/32333870
# answered by SO user Siyh: https://stackoverflow.com/a/35705477
def scale_bar(ax, length=20, location=(0.92, 0.95)):
    """
    Create a scale bar in a cartopy GeoAxes.

    Parameters
    ----------

    ax : cartopy.mpl.geoaxes.GeoAxes
        the cartopy GeoAxes to add the scalebar to.

    length : int, float (default 20)
        the length of the scalebar, in km

    location : tuple(float, float) (default (0.92, 0.95))
        the location of the center right corner of the scalebar, in fractions of the axis.

    Returns
    -------
    ax : cartopy.mpl.geoaxes.GeoAxes
        the cartopy GeoAxes object

    """
    x0, x1, y0, y1 = ax.get_extent() # get the current extent of the axis
    sbx = x0 + (x1 - x0) * location[0] # get the right x coordinate of the scale bar
    sby = y0 + (y1 - y0) * location[1] # get the right y coordinate of the scale bar

    ax.plot([sbx, sbx-length*1000], [sby, sby], color='k', linewidth=4, transform=ax.projection) # plot a thick black line
    ax.plot([sbx-(length/2)*1000, sbx-length*1000], [sby, sby], color='w', linewidth=2, transform=ax.projection) # plot a white line from 0 to halfway

    ax.text(sbx, sby-(length/4)*1000, f"{length} km", ha='center', transform=ax.projection, fontsize=6) # add a label at the right side
    ax.text(sbx-(length/2)*1000, sby-(length/4)*1000, f"{int(length/2)} km", ha='center', transform=ax.projection, fontsize=6) # add a label in the center
    ax.text(sbx-length*1000, sby-(length/4)*1000, '0 km', ha='center', transform=ax.projection, fontsize=6) # add a label at the left side

    return ax



In [ ]:
ni_utm=ccrs.UTM(29) #create Universal Transverse Mercator reference system to transform data # create cartopy CRS representation of CRS associated with the outline dataset

fig = plt.figure(figsize=(8, 8))  # create a figure of size 8x8 (representing the page size in inches)
ax = plt.axes(projection=ni_utm)  # create an axes object in the figure, using a UTM projection,
# where we can actually plot our data.

In [ ]:
%matplotlib widget

In [ ]:
# first, we just add the outline of data zones using cartopy's ShapelyFeature
zone_feature = ShapelyFeature(census_zones['geometry'], ni_utm, edgecolor='k', facecolor='w')
ax.add_feature(zone_feature) # add the features we've created to the map.

In [ ]:
import folium

In [ ]:
m = census_zones.explore('PopnDens', cmap='viridis')

m # show the map

In [ ]:
census_zones.head()

Website to determine UTM zone: https://mangomap.com/robertyoung/maps/69585/what-utm-zone-am-i-in-# 

In [ ]:
ni_utm = ccrs.UTM(29) #Universal Transverse Mercator reference system Northern Ireland lies in zone 29

Unemployment

In [ ]:
# unemployment rate 
# use economically active - unemployed / (employed + unemployed) * 100 


In [ ]:
# map of unemployment rate per data zone